In [ ]:
#Install Docling
#!pip install -q docling

In [ ]:
from pathlib import Path
from docling.document_converter import DocumentConverter

converter = DocumentConverter()


def extract_pdf_structure(pdf_path: Path):
    """
    Extract the structured representation of a guideline PDF
    using Docling.
    """
    result = converter.convert(pdf_path)
    return result.document
    

In [ ]:
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

# Use the same tokenizer as your embedding model
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

# Create Docling's hybrid chunker
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=480,
    merge_peers=True
)

In [ ]:
# The following lines will automatically find PDFs inside the four folders
KB_DIR = Path("Knowledge Base")

SPECIALTIES = [
    "Cardiology",
    "Infectious Diseases",
    "Nephrology",
    "Pulmonology",
]

pdf_files = sorted(
    pdf_path
    for specialty in SPECIALTIES
    for pdf_path in (KB_DIR / specialty).rglob("*.pdf")
    if ".ipynb_checkpoints" not in pdf_path.parts
)

print(f"PDF files found: {len(pdf_files)}")

In [ ]:
def clean_kb_chunks(chunks):
    """
    Minimal cleaning of Docling HybridChunker output.
    For now, only remove Table of Contents chunks.
    """
    cleaned_chunks = []

    for chunk in chunks:
        headings = chunk.meta.headings or []

        # Remove chunks belonging to the Table of Contents
        if any("TABLE OF CONTENTS" in heading.upper() for heading in headings):
            continue

        cleaned_chunks.append(chunk)

    return cleaned_chunks


In [ ]:
NON_CLINICAL_PATTERNS = {
    "table of contents",
    "author",
    "writing committee",
    "relationships with industry",
    "conflict of interest",
    "document review and approval",
    "acknowledgment",
    "references",
}


def remove_non_clinical_chunks(chunks):
    """
    Conservatively remove chunks belonging to clearly
    non-clinical sections based on heading metadata.
    """
    cleaned_chunks = []

    for chunk in chunks:
        headings = chunk.meta.headings or []

        is_non_clinical = any(
            pattern in heading.lower()
            for heading in headings
            for pattern in NON_CLINICAL_PATTERNS
        )

        if is_non_clinical:
            continue

        cleaned_chunks.append(chunk)

    return cleaned_chunks


In [ ]:

import json
from pathlib import Path

KB_DIR = Path("Knowledge Base")
OUTPUT_DIR = KB_DIR / "Processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def process_specialty(specialty):
    """
    Parse, chunk, clean, and save all guideline PDFs
    belonging to one specialty.
    """

    specialty_dir = KB_DIR / specialty

    pdf_files = sorted(
        pdf_path
        for pdf_path in specialty_dir.rglob("*.pdf")
        if ".ipynb_checkpoints" not in pdf_path.parts
    )

    print(f"\nSpecialty: {specialty}")
    print(f"PDF files found: {len(pdf_files)}")

    specialty_chunks = []

    for pdf_path in pdf_files:

        print(f"Processing: {pdf_path.name}")

        # 1. Parse document
        document = extract_pdf_structure(pdf_path)

        # 2. Hybrid chunking
        chunks = list(chunker.chunk(document))

        # 3. Remove Table of Contents
        chunks = clean_kb_chunks(chunks)

        # 4. Remove clearly non-clinical content
        chunks = remove_non_clinical_chunks(chunks)

        # 5. Prepare chunks for saving
        for chunk in chunks:

            specialty_chunks.append({
                "text": chunk.text,
                "specialty": specialty,
                "source_file": pdf_path.name,
                "source_path": str(pdf_path),
                "headings": chunk.meta.headings or [],
            })

    # Create safe filename
    specialty_filename = (
        specialty.lower()
        .replace(" ", "_")
        + "_chunks.jsonl"
    )

    output_file = OUTPUT_DIR / specialty_filename

    # Save immediately after this specialty is complete
    with output_file.open("w", encoding="utf-8") as f:
        for i, record in enumerate(specialty_chunks, start=1):

            record["chunk_id"] = (
                f"{specialty.lower().replace(' ', '_')}"
                f"_chunk_{i:06d}"
            )

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Processing summary
    print("\n" + "=" * 80)
    print(f"{specialty.upper()} PROCESSING COMPLETE")
    print("=" * 80)
    print(f"PDF files processed: {len(pdf_files)}")
    print(f"Final chunks: {len(specialty_chunks)}")
    print(f"Saved to: {output_file}")

    return specialty_chunks

In [ ]:
cardiology_chunks = process_specialty("Cardiology")

In [ ]:
infectious_diseases_chunks = process_specialty(
    "Infectious Diseases"
)

In [ ]:
nephrology_chunks = process_specialty("Nephrology")

In [ ]:
pulmonology_chunks = process_specialty("Pulmonology")

In [ ]:
# ============================================================
# Diagnostic only: how many KB chunks look like reference lists?
# Reads the existing processed files; changes nothing.
# ============================================================
import json, re
from pathlib import Path
from collections import Counter

PROC = Path("Knowledge Base/Processed")
FILES = ["cardiology_chunks.jsonl", "infectious_diseases_chunks.jsonl",
         "nephrology_chunks.jsonl", "pulmonology_chunks.jsonl"]

CITE_RE = [re.compile(p, re.I) for p in [
    r"\bet al\b",                        # author lists
    r"\bdoi\b|10\.\d{4,9}/\S+",          # DOIs
    r"\bPMID\b|\bPMCID\b",               # PubMed IDs
    r"\b(?:19|20)\d{2}\s*;\s*\d+",       # year;volume, e.g. 2019;140
    r"\b\d+\s*\(\d+\)\s*:\s*e?\d+",      # volume(issue):page, e.g. 140(11):e596
]]

def cite_density(text):
    """Citation-pattern matches per 100 words."""
    hits = sum(len(r.findall(text)) for r in CITE_RE)
    return 100 * hits / max(len(text.split()), 1)

chunks = []
for fn in FILES:
    with open(PROC / fn, encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            c["density"] = cite_density(c["text"])
            chunks.append(c)

print(f"Total KB chunks: {len(chunks):,}\n")
print("Chunks at or above each density threshold:")
for t in (1, 2, 3, 5, 8):
    by_spec = Counter(c["specialty"] for c in chunks if c["density"] >= t)
    print(f"  >= {t}: {sum(by_spec.values()):>5,}  {dict(by_spec)}")

print("\nMost common last heading among chunks with density >= 3:")
heads = Counter((c["headings"][-1] if c["headings"] else "(no heading)")
                for c in chunks if c["density"] >= 3)
for h, n in heads.most_common(10):
    print(f"  {n:>4}  {h[:80]}")

def show(lo, hi, k=2):
    sample = [c for c in chunks if lo <= c["density"] < hi][:k]
    for c in sample:
        print("-" * 80)
        print(f"{c['chunk_id']} | density {c['density']:.1f} | headings {c['headings'][-1:] if c['headings'] else None}")
        print(c["text"][:400])

print("\n=== Samples, density 1-3 (borderline) ===");  show(1, 3)
print("\n=== Samples, density 3-5 ===");              show(3, 5)
print("\n=== Samples, density >= 5 (likely references) ==="); show(5, 1e9)

In [ ]:
# Diagnostic only: every chunk with density between 2 and 3
mid = sorted((c for c in chunks if 2 <= c["density"] < 3), key=lambda c: c["density"])
print(f"Chunks with density 2-3: {len(mid)}\n")
for c in mid:
    text = " ".join(c["text"].split())
    print(f"{c['density']:.1f} | {c['chunk_id']} | {text[:150]}")

In [ ]:
# Diagnostic only: density >= 2 AND a numbered citation entry
ENTRY_RE = re.compile(
    r"(?:^|\s)(?:-\s*|\[)?(?:S[\d.]+-)?\d{1,4}\.?\s+"             # list number: "83.", "S6.4.2-17.", "- 168"
    r"[A-Z][A-Za-z'\-]+(?:[ -][A-Z][A-Za-z'\-]+)?\s+[A-Z]{1,3}\s?[,.]"  # surname + initials: "Patel MR,"
)

def is_reference(c, min_density=2):
    return c["density"] >= min_density and ENTRY_RE.search(c["text"]) is not None

flagged = [c for c in chunks if is_reference(c)]
print(f"Flagged as references: {len(flagged):,}  {dict(Counter(c['specialty'] for c in flagged))}")

# 1) High-density chunks the rule KEEPS: are any of these references we miss?
kept_high = [c for c in chunks if c["density"] >= 3 and not is_reference(c)]
print(f"\nDensity >= 3 but KEPT: {len(kept_high)}")
for c in kept_high[:40]:
    print(f"{c['density']:.1f} | {c['chunk_id']} | {' '.join(c['text'].split())[:150]}")

# 2) Density 2-3 chunks the rule REMOVES: should all be references
mid_removed = [c for c in chunks if 2 <= c["density"] < 3 and is_reference(c)]
print(f"\nDensity 2-3 REMOVED: {len(mid_removed)}")
for c in mid_removed:
    print(f"{c['density']:.1f} | {c['chunk_id']} | {' '.join(c['text'].split())[:100]}")

In [ ]:
# Diagnostic only: count full journal citations (year;volume:page) per chunk
JCITE_RE = re.compile(
    r"\b(?:19|20)\d{2}\s*;\s*\d+\s*(?:\(\s*[\w\-]+\s*\))?\s*:\s*[A-Za-z]?\d+"
)   # 2011;171:134 | 2024 ; 79(9): 2306 | 2010; 50:291 | 2019;74:i1

for c in chunks:
    c["n_cites"] = len(JCITE_RE.findall(c["text"]))

def is_reference_v2(c, min_cites=2):
    return c["n_cites"] >= min_cites

for k in (1, 2, 3):
    by = Counter(c["specialty"] for c in chunks if c["n_cites"] >= k)
    print(f"n_cites >= {k}: {sum(by.values()):,}  {dict(by)}")

# 1) Clinical chunks seen in earlier outputs: all must be KEPT
CLINICAL_IDS = [
    "infectious_diseases_chunk_000370",   # sulbactam-durlobactam dosing
    "cardiology_chunk_001915",            # CARPREG II risk score
    "cardiology_chunk_005258",            # SMAA text
    "pulmonology_chunk_001029",           # PEEP trial
    "pulmonology_chunk_001087",           # NIV after abdominal surgery
    "pulmonology_chunk_001104",           # NIV after extubation
    "nephrology_chunk_000143",            # tonsillectomy recommendation
    "nephrology_chunk_001222",            # vancomycin pharmacist
    "infectious_diseases_chunk_003226",   # Lenze / Reis follow-up
    "cardiology_chunk_000928",            # caption, Fravel et al.
    "cardiology_chunk_004025",            # caption, Mangrum and DiMarco
]
by_id = {c["chunk_id"]: c for c in chunks}
print("\n1) Known clinical chunks:")
for cid in CLINICAL_IDS:
    c = by_id[cid]
    print(f"  {'REMOVED!' if is_reference_v2(c) else 'kept    '} | {c['n_cites']} cites | {cid}")

# 2) Removed chunks with the FEWEST citations per word: mixed clinical + reference chunks?
flagged2 = [c for c in chunks if is_reference_v2(c)]
low = sorted(flagged2, key=lambda c: c["n_cites"] / max(len(c["text"].split()), 1))[:25]
print(f"\n2) Removed: {len(flagged2):,}. The 25 with the lowest citation share:")
for c in low:
    print(f"  {c['n_cites']:>2} cites / {len(c['text'].split()):>3} words | {c['chunk_id']} | {' '.join(c['text'].split())[:130]}")

# 3) High-density chunks still kept: should now be few, mostly captions
kept_high2 = [c for c in chunks if c["density"] >= 3 and not is_reference_v2(c)]
print(f"\n3) Density >= 3 but KEPT: {len(kept_high2)}")
for c in kept_high2[:30]:
    print(f"  {c['density']:.1f} | {c['n_cites']} cites | {c['chunk_id']} | {' '.join(c['text'].split())[:130]}")

In [ ]:
# Diagnostic only: density >= 3 AND at least 1 full journal citation
def is_reference_v3(c):
    return c["density"] >= 3 and c["n_cites"] >= 1

flagged3 = [c for c in chunks if is_reference_v3(c)]
print(f"Removed by v3: {len(flagged3):,}  {dict(Counter(c['specialty'] for c in flagged3))}")

# 1) Known clinical chunks, including the KDIGO ones from the last run: all must be KEPT
MIXED_IDS = ["nephrology_chunk_001902", "nephrology_chunk_002204", "nephrology_chunk_002112",
             "nephrology_chunk_002207", "nephrology_chunk_001766", "nephrology_chunk_002119"]
print("\n1) Known clinical chunks:")
for cid in CLINICAL_IDS + MIXED_IDS:
    c = by_id[cid]
    print(f"  {'REMOVED!' if is_reference_v3(c) else 'kept    '} | density {c['density']:.1f} | {c['n_cites']} cites | {cid}")

# 2) Removed chunks closest to the cut-off: should all be references
low3 = sorted(flagged3, key=lambda c: c["density"])[:25]
print("\n2) Removed, lowest density first:")
for c in low3:
    print(f"  {c['density']:.1f} | {c['n_cites']} cites | {c['chunk_id']} | {' '.join(c['text'].split())[:120]}")

# 3) Chunks with >= 2 citations that are now KEPT: clinical/mixed, or references we miss?
kept_cited = [c for c in chunks if c["n_cites"] >= 2 and not is_reference_v3(c)]
print(f"\n3) >= 2 citations but KEPT: {len(kept_cited)}")
for c in kept_cited[:30]:
    print(f"  {c['density']:.1f} | {c['n_cites']} cites | {c['chunk_id']} | {' '.join(c['text'].split())[:120]}")

In [ ]:
# ============================================================
# Remove reference-list chunks from the KB (post-processing)
# Rule: citation density >= 3 per 100 words AND >= 1 full journal citation
# Original chunk_ids are kept, so smoke-test labels stay valid.
# ============================================================
import json, re, csv
from pathlib import Path

IN_DIR  = Path("Knowledge Base/Processed")
OUT_DIR = Path("Knowledge Base/Processed_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = ["cardiology_chunks.jsonl", "infectious_diseases_chunks.jsonl",
         "nephrology_chunks.jsonl", "pulmonology_chunks.jsonl"]

CITE_RE = [re.compile(p, re.I) for p in [
    r"\bet al\b",
    r"\bdoi\b|10\.\d{4,9}/\S+",
    r"\bPMID\b|\bPMCID\b",
    r"\b(?:19|20)\d{2}\s*;\s*\d+",
    r"\b\d+\s*\(\d+\)\s*:\s*e?\d+",
]]
JCITE_RE = re.compile(
    r"\b(?:19|20)\d{2}\s*;\s*\d+\s*(?:\(\s*[\w\-]+\s*\))?\s*:\s*[A-Za-z]?\d+"
)
MIN_DENSITY, MIN_JCITES = 3, 1

def cite_density(text):
    hits = sum(len(r.findall(text)) for r in CITE_RE)
    return 100 * hits / max(len(text.split()), 1)

def is_reference_chunk(text):
    return (cite_density(text) >= MIN_DENSITY
            and len(JCITE_RE.findall(text)) >= MIN_JCITES)

removed_log, total_kept = [], 0
for fn in FILES:
    kept, removed = [], 0
    with open(IN_DIR / fn, encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            if is_reference_chunk(c["text"]):
                removed += 1
                removed_log.append({
                    "chunk_id": c["chunk_id"],
                    "specialty": c["specialty"],
                    "source_file": c["source_file"],
                    "density": round(cite_density(c["text"]), 2),
                    "n_cites": len(JCITE_RE.findall(c["text"])),
                })
            else:
                kept.append(c)
    with open(OUT_DIR / fn, "w", encoding="utf-8") as f:
        for c in kept:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    total_kept += len(kept)
    print(f"{fn:36s} kept {len(kept):>6,} | removed {removed:>5,}")

with open(OUT_DIR / "removed_reference_chunks.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(removed_log[0].keys()))
    w.writeheader()
    w.writerows(removed_log)

print(f"\nTotal kept: {total_kept:,} | removed: {len(removed_log):,}")
print("Log saved:", OUT_DIR / "removed_reference_chunks.csv")